# 06 — Sparse Autoencoders (SAE) — Contribution C1

Mechanistic interpretability of **DeiT-Base** and **DINOv2-ViT-B/14**
on the CRC-VAL-HE-7K histology dataset (9 classes).

## Pipeline

```
Trained ViT
  → Collect MLP activations (blocks.8.mlp)
  → Train TopK-SAE (Gao et al., 2024)
  → Analyse features: top-K activating images per feature
  → Steerability: ablate/amplify features, measure class-probability shift
  → Save results + upload to Drive
```

**References**
- Cunningham et al. ICLR 2024 — "SAEs Find Highly Interpretable Features"
- Templeton et al. 2024 — "Scaling Monosemanticity" (Anthropic)
- Gao et al. 2024 — "Scaling and Evaluating SAEs" (OpenAI; TopK)

**Edit only** `USER CONFIG` to switch between models.

**Environment**: Kaggle GPU T4 / P100 — Phase 1 only.

In [ ]:
# 0. Kaggle setup
!rm -rf /kaggle/working/xai-vit-medical
!git clone https://github.com/youssef-nouiouar/xai-vit-medical.git /kaggle/working/xai-vit-medical
!pip install -q timm albumentations loguru
!pip install -q PyDrive2
!pip install -q scikit-learn

In [ ]:
from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials

auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

In [ ]:
# ============================================================
# USER CONFIG — edit this cell only
# ============================================================

# Primary SAE targets: deit_base | dinov2
MODEL_NAME = "deit_base"

# Google Drive folder containing the checkpoint
DRIVE_FOLDER_ID = "1eq-Jt6O6gO0Ck_oQYwmmc2qrCVLfKlec"  # <-- update as needed

# ── Activation collection ──
MAX_BATCHES_COLLECT = 150    # 150 × 32 = 4 800 images ≈ 500 images/classe
COLLECT_BATCH_SIZE  = 32
EXTRACT_LAYERS      = [6, 8, 10]   # DeiT: [6,8,10]; DINOv2: [8,10]
EXTRACT_SITE        = "residual"   # "mlp" | "residual" | "attn"
TOKEN_SELECTION     = "patches"    # "patches" | "cls" | "all"

# ── SAE architecture (TopK — Gao et al. 2024) ──
D_SAE    = 4096   # 4096 recommandé (×5.3); 6144 si production
K_SPARSE = 64     # features actives par token — L0 garanti = k

# ── SAE training ──
TRAIN_BATCH_SIZE = 4096
NUM_EPOCHS       = 30
LEARNING_RATE    = 5e-5
WARMUP_STEPS     = 500
AUX_COEFF        = 1 / 32   # coefficient aux_loss (Gao et al. 2024) — ressuscite les features mortes

# ── Feature analysis ──
TOP_K_IMAGES_PER_FEATURE = 16
N_FEATURES_VISUALIZE     = 32

# ── Steerability ──
N_STEER_IMAGES   = 50
STEER_BATCH_SIZE = 8

# ── General ──
IMAGE_SIZE  = 224
NUM_WORKERS = 2
SEED        = 42
CACHE_ACTS  = True

# Paths (Kaggle)
TRAINVAL_ROOT_STR = "/kaggle/input/datasets/youssefnouiouar1/colorectal-cancer-histology-nct-crc-he-100k-and-7k/NCT-CRC-HE-100K/NCT-CRC-HE-100K"
TEST_ROOT_STR     = "/kaggle/input/datasets/youssefnouiouar1/colorectal-cancer-histology-nct-crc-he-100k-and-7k/CRC-VAL-HE-7K/CRC-VAL-HE-7K"
PROJECT_ROOT      = "/kaggle/working/xai-vit-medical"

In [ ]:
# ============================================================
# Per-model SAE configuration
# ============================================================

SAE_MODEL_CONFIGS = {
    "deit_base": {
        "source"          : "timm",
        "timm_name"       : "deit_base_patch16_224",
        "checkpoint_fname": "deit_base_best.pth",
        "blocks_prefix"   : "blocks",          # model.blocks
        "extract_layers"  : [6, 8, 10],        # default; override via EXTRACT_LAYERS
        "extract_site"    : "residual",
        "d_in"            : 768,
        "num_extra_tokens": 1,                 # CLS only
        "n_patch_tokens"  : 196,               # 14×14
        "patch_size"      : 16,
    },
    "dinov2": {
        "source"          : "torch_hub",
        "timm_name"       : None,
        "checkpoint_fname": "dinov2_best.pth",
        "blocks_prefix"   : "backbone.blocks", # DINOv2Classifier: model.backbone.blocks
        "extract_layers"  : [8, 10],
        "extract_site"    : "mlp",
        "d_in"            : 768,
        "num_extra_tokens": 1,                 # CLS only
        "n_patch_tokens"  : 256,               # 16×16
        "patch_size"      : 14,
    },
}

assert MODEL_NAME in SAE_MODEL_CONFIGS, (
    f"Unknown MODEL_NAME: {MODEL_NAME}. Choose from {list(SAE_MODEL_CONFIGS)}"
)
MCFG = SAE_MODEL_CONFIGS[MODEL_NAME]
D_IN = MCFG["d_in"]

# Build the module path used by the steerability hook
_prefix = MCFG["blocks_prefix"]
TARGET_LAYER_PATH = (
    f"{_prefix}.{EXTRACT_LAYERS[0]}.{EXTRACT_SITE}"
    if EXTRACT_SITE != "residual"
    else f"{_prefix}.{EXTRACT_LAYERS[0]}"
)

print(f"Model            : {MODEL_NAME}")
print(f"Extract layers   : {EXTRACT_LAYERS}  (site='{EXTRACT_SITE}')")
print(f"Target layer path: {TARGET_LAYER_PATH}")
print(f"d_in             : {D_IN}")
print(f"d_sae            : {D_SAE}  (expansion ×{D_SAE/D_IN:.1f})")
print(f"k                : {K_SPARSE}")
print(f"Token selection  : {TOKEN_SELECTION} ({MCFG['n_patch_tokens']} patch tokens/image)")

In [ ]:
import csv
import gc
import json
import os
import sys
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
from PIL import Image
from torch.utils.data import DataLoader, Subset, TensorDataset
from tqdm.notebook import tqdm

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.data.crc_dataset import CRCHistologyDataset, CRCSplits, DEFAULT_CRC_CLASSES
from src.utils.seed import set_seed

set_seed(SEED)
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CLASS_NAMES = list(DEFAULT_CRC_CLASSES)
NUM_CLASSES = len(CLASS_NAMES)

SAVE_DIR  = Path(f"{PROJECT_ROOT}/outputs/sae/{MODEL_NAME}")
CACHE_DIR = Path(f"{PROJECT_ROOT}/outputs/cache/{MODEL_NAME}")
for d in (SAVE_DIR, CACHE_DIR):
    d.mkdir(parents=True, exist_ok=True)

TRAINVAL_ROOT = Path(TRAINVAL_ROOT_STR)
TEST_ROOT     = Path(TEST_ROOT_STR)
CKPT_LOCAL    = "/kaggle/input/models/youssefnouiouar1/crt-deit/pytorch/default/1/deit_base_patch16_best.pth"

print(f"Checkpoint : {CKPT_LOCAL}")
print(f"Device     : {DEVICE}")
print(f"Classes    : {CLASS_NAMES}")
print(f"Output     : {SAVE_DIR}")

In [ ]:
# =============================================================================
# SAE CLASSES — self-contained, no src.xai dependencies
# ActivationExtractor | SparseAutoencoder | SAETrainer | FeatureAnalyzer
# =============================================================================


class ActivationExtractor:
    """Extracts intermediate activations from a ViT via PyTorch forward hooks."""

    def __init__(self, model, model_name: str, layers: list, site: str = "mlp"):
        self.model       = model
        self.activations = defaultdict(list)
        self.hooks       = []
        self._register_hooks(model_name, layers, site)

    def _get_blocks(self, model_name: str):
        if model_name == "deit_base":
            return self.model.blocks
        elif model_name == "dinov2":
            return self.model.backbone.blocks
        else:
            raise ValueError(f"Unsupported model: {model_name}")

    def _register_hooks(self, model_name: str, layers: list, site: str):
        blocks = self._get_blocks(model_name)
        for layer_idx in layers:
            block = blocks[layer_idx]
            if site == "residual":
                hook = block.register_forward_hook(
                    self._make_hook(f"layer_{layer_idx}_residual"))
            elif site == "mlp":
                hook = block.mlp.register_forward_hook(
                    self._make_hook(f"layer_{layer_idx}_mlp"))
            elif site == "attn":
                hook = block.attn.register_forward_hook(
                    self._make_hook(f"layer_{layer_idx}_attn"))
            else:
                raise ValueError(f"Unknown site: {site}")
            self.hooks.append(hook)

    def _make_hook(self, name: str):
        def hook_fn(module, input, output):
            if isinstance(output, tuple):
                output = output[0]
            self.activations[name].append(output.detach().cpu())
        return hook_fn

    def extract(self, dataloader, token_mode: str = "patches", device: str = "cuda") -> dict:
        self.model.to(device)
        self.activations.clear()
        with torch.no_grad():
            for batch_images, _ in dataloader:
                batch_images = batch_images.to(device)
                _ = self.model(batch_images)
        result = {}
        for name, act_list in self.activations.items():
            all_acts = torch.cat(act_list, dim=0)
            if token_mode == "cls":
                all_acts = all_acts[:, 0, :]
            elif token_mode == "patches":
                all_acts = all_acts[:, 1:, :].reshape(-1, all_acts.shape[-1])
            elif token_mode == "all":
                all_acts = all_acts.reshape(-1, all_acts.shape[-1])
            result[name] = all_acts
        self.remove_hooks()
        return result

    def remove_hooks(self):
        for h in self.hooks:
            h.remove()
        self.hooks.clear()


class SparseAutoencoder(nn.Module):
    """
    TopK Sparse Autoencoder + Auxiliary loss (Gao et al., 2024 — OpenAI).

    Encoder : pre_bias → W_enc → TopK(k) → codes
    Decoder : codes   → W_dec → + b_dec  → x_hat

    aux_loss : force les features mortes à reconstruire l'erreur résiduelle.
        1. Calcule les pré-activations pour TOUTES les features (pas de TopK)
        2. Applique ReLU uniquement sur les features MORTES → ghost_acts
        3. Reconstruit le résiduel (x - x_hat) avec ces ghost_acts
        4. aux_loss = MSE(résiduel, x_hat_ghost)
        Résultat : les features mortes reçoivent un gradient → elles se réactivent.
    """

    def __init__(self, d_input: int, d_sae: int = 4096, k: int = 64):
        super().__init__()
        self.d_input  = d_input
        self.d_hidden = d_sae
        self.k        = k
        self.W_enc = nn.Parameter(torch.empty(d_input, d_sae))
        self.b_pre = nn.Parameter(torch.zeros(d_input))
        self.b_enc = nn.Parameter(torch.zeros(d_sae))
        self.W_dec = nn.Parameter(torch.empty(d_sae, d_input))
        self.b_dec = nn.Parameter(torch.zeros(d_input))
        self._init_weights()

    def _init_weights(self):
        nn.init.kaiming_uniform_(self.W_enc)
        nn.init.kaiming_uniform_(self.W_dec)
        with torch.no_grad():
            self.W_dec.data = F.normalize(self.W_dec.data, dim=-1)

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        x_centered          = x - self.b_pre
        pre_acts            = x_centered @ self.W_enc + self.b_enc
        topk_vals, topk_idx = pre_acts.topk(self.k, dim=-1)
        codes               = torch.zeros_like(pre_acts)
        codes.scatter_(-1, topk_idx, F.relu(topk_vals))
        return codes

    def decode(self, codes: torch.Tensor) -> torch.Tensor:
        return codes @ self.W_dec + self.b_dec

    def forward(self, x: torch.Tensor, dead_mask: torch.Tensor = None):
        """
        dead_mask : (d_sae,) bool tensor, True = feature morte.
                    Si None → aux_loss = 0 (pas de pénalité).
        """
        codes   = self.encode(x)
        x_hat   = self.decode(codes)
        l2_loss = F.mse_loss(x_hat, x)

        # ── Auxiliary loss (Gao et al. 2024) ──────────────────────────────
        # But : donner un gradient aux features mortes via "ghost gradients"
        if dead_mask is not None and dead_mask.any():
            x_centered  = x - self.b_pre
            pre_acts    = x_centered @ self.W_enc + self.b_enc   # (B, d_sae)
            # ReLU sur features mortes uniquement (pas de contrainte TopK ici)
            ghost_acts  = F.relu(pre_acts) * dead_mask.float()   # (B, d_sae)
            x_hat_ghost = ghost_acts @ self.W_dec + self.b_dec   # (B, d_input)
            # Cible : ce que le SAE normal n'arrive pas à reconstruire
            residual    = (x - x_hat).detach()                   # stop gradient
            aux_loss    = F.mse_loss(residual, x_hat_ghost)
        else:
            aux_loss = torch.zeros(1, device=x.device)

        return x_hat, codes, l2_loss, aux_loss

    @torch.no_grad()
    def normalize_decoder(self):
        self.W_dec.data = F.normalize(self.W_dec.data, dim=-1)


class SAETrainer:
    """
    Entraîne SparseAutoencoder avec :
    - LR warmup linéaire
    - Gradient clipping
    - Suivi EMA des features mortes → passage du dead_mask au forward
    """

    def __init__(self, sae: SparseAutoencoder, config: dict, device: str = "cuda"):
        self.sae            = sae.to(device)
        self.config         = config
        self.device         = device
        self.aux_coeff      = config.get("aux_coeff", 1 / 32)
        self.optimizer      = torch.optim.Adam(
            sae.parameters(), lr=config["lr"], betas=(0.9, 0.999)
        )
        self.step           = 0
        # EMA de l'activité des features pour détecter les mortes
        self._ema_decay     = 0.99
        self._feat_activity = torch.zeros(sae.d_hidden)  # (d_sae,) CPU

    @property
    def dead_mask(self) -> torch.Tensor:
        """True pour les features dont l'activité EMA est quasi nulle."""
        return self._feat_activity < 1.0   # CPU bool tensor

    def _update_activity(self, f: torch.Tensor):
        """Met à jour l'EMA d'activité après chaque batch."""
        active              = (f > 0).float().sum(0).cpu()   # nb activations dans le batch
        self._feat_activity = (
            self._ema_decay * self._feat_activity
            + (1 - self._ema_decay) * active
        )

    def _warmup_lr(self):
        if self.step < self.config["warmup_steps"]:
            lr = self.config["lr"] * (self.step / max(self.config["warmup_steps"], 1))
            for pg in self.optimizer.param_groups:
                pg["lr"] = lr

    def train(self, activations: torch.Tensor) -> list[dict]:
        from torch.utils.data import TensorDataset
        dataset    = TensorDataset(activations)
        dataloader = DataLoader(
            dataset, batch_size=self.config["batch_size"], shuffle=True, drop_last=True
        )
        history = []

        for epoch in range(self.config["num_epochs"]):
            metrics = {"l2": 0., "aux": 0., "total": 0., "l0": 0., "r2": 0., "dead": 0., "n": 0}

            for (batch,) in dataloader:
                batch = batch.to(self.device)
                self.step += 1
                self._warmup_lr()

                # Passer le dead_mask pour activer l'aux_loss sur features mortes
                dm = self.dead_mask.to(self.device)
                x_hat, f, l2_loss, aux_loss = self.sae(batch, dead_mask=dm)
                total_loss = l2_loss + self.aux_coeff * aux_loss

                self.optimizer.zero_grad()
                total_loss.backward()
                torch.nn.utils.clip_grad_norm_(self.sae.parameters(), max_norm=1.0)
                self.optimizer.step()
                self.sae.normalize_decoder()

                # Mettre à jour l'EMA APRÈS le step optimizer
                self._update_activity(f.detach())

                with torch.no_grad():
                    l0    = (f > 0).float().sum(-1).mean().item()
                    x_var = ((batch - batch.mean(0)) ** 2).mean().item()
                    r2    = max(0., 1. - l2_loss.item() / (x_var + 1e-8))
                    n_dead = int(self.dead_mask.sum().item())
                    metrics["l2"]    += l2_loss.item()
                    metrics["aux"]   += aux_loss.item()
                    metrics["total"] += total_loss.item()
                    metrics["l0"]    += l0
                    metrics["r2"]    += r2
                    metrics["dead"]  += n_dead
                    metrics["n"]     += 1

            n   = metrics["n"]
            avg = {k: v / n for k, v in metrics.items() if k != "n"}
            history.append(avg)
            print(
                f"Epoch {epoch+1:3d}/{self.config['num_epochs']} | "
                f"L2: {avg['l2']:.6f} | aux: {avg['aux']:.6f} | "
                f"L0: {avg['l0']:.1f}/{self.sae.k} | "
                f"R²: {avg['r2']:.3f} | "
                f"Dead: {int(avg['dead'])}/{self.sae.d_hidden} "
                f"({avg['dead']/self.sae.d_hidden*100:.1f}%)"
            )
        return history


class FeatureAnalyzer:
    """Analysis and manipulation of SAE features for histopathology interpretability."""

    def __init__(self, sae: SparseAutoencoder, device: str = "cuda"):
        self.sae    = sae.to(device)
        self.device = device

    def compute_top_activating(
        self,
        activations: torch.Tensor,
        image_indices: torch.Tensor,
        top_k_images: int = 16,
        batch_size: int = 2048,
    ) -> dict:
        self.sae.eval()
        all_codes_list = []
        with torch.no_grad():
            for i in range(0, len(activations), batch_size):
                batch = activations[i : i + batch_size].to(self.device)
                all_codes_list.append(self.sae.encode(batch).cpu())
        all_codes = torch.cat(all_codes_list, dim=0)

        n_images = int(image_indices.max().item()) + 1
        d_sae    = all_codes.shape[1]
        img_max  = torch.zeros(n_images, d_sae)
        idx_exp  = image_indices.unsqueeze(1).expand(-1, d_sae)
        img_max.scatter_reduce_(0, idx_exp, all_codes, reduce="amax", include_self=True)

        feature_data = {}
        for feat_idx in range(d_sae):
            feat_img_acts     = img_max[:, feat_idx]
            k                 = min(top_k_images, n_images)
            top_vals, top_idx = feat_img_acts.topk(k)
            nonzero           = top_vals > 0
            feat_token_acts   = all_codes[:, feat_idx]
            freq              = (feat_token_acts > 0).float().mean().item()
            pos_acts          = feat_token_acts[feat_token_acts > 0]
            mean_act          = pos_acts.mean().item() if len(pos_acts) > 0 else 0.0
            feature_data[feat_idx] = {
                "top_image_indices"   : top_idx[nonzero],
                "activation_strengths": top_vals[nonzero],
                "activation_frequency": freq,
                "mean_activation"     : mean_act,
            }
        return feature_data

    def feature_class_association(
        self, activations: torch.Tensor, labels: torch.Tensor,
        num_classes: int, batch_size: int = 2048,
    ) -> torch.Tensor:
        self.sae.eval()
        all_feats = []
        with torch.no_grad():
            for i in range(0, len(activations), batch_size):
                batch = activations[i : i + batch_size].to(self.device)
                all_feats.append(self.sae.encode(batch).cpu())
        features    = torch.cat(all_feats, dim=0)
        class_means = torch.zeros(num_classes, features.shape[1])
        for c in range(num_classes):
            mask = labels == c
            if mask.sum() > 0:
                class_means[c] = features[mask].mean(dim=0)
        return class_means

    def steer_activation(
        self, activation: torch.Tensor, feature_idx: int, scale: float = 2.0
    ) -> torch.Tensor:
        self.sae.eval()
        with torch.no_grad():
            f = self.sae.encode(activation.to(self.device))
            f[:, feature_idx] *= scale
            modified = self.sae.decode(f)
        return modified.cpu()

    def ablate_feature(self, activation: torch.Tensor, feature_idx: int) -> torch.Tensor:
        return self.steer_activation(activation, feature_idx, scale=0.0)

    def compute_health_metrics(self, activations: torch.Tensor) -> dict:
        self.sae.eval()
        all_features = []
        total_mse, total_var, n = 0., 0., 0
        batch_size = 4096
        with torch.no_grad():
            for i in range(0, len(activations), batch_size):
                batch = activations[i : i + batch_size].to(self.device)
                x_hat, f, l2, _ = self.sae(batch)
                all_features.append((f > 0).float().cpu())
                total_mse += ((x_hat - batch) ** 2).sum().item()
                total_var += ((batch - batch.mean(0)) ** 2).sum().item()
                n         += len(batch)
        all_features  = torch.cat(all_features, dim=0)
        ever_active   = all_features.sum(dim=0) > 0
        num_dead      = (~ever_active).sum().item()
        l0_per_sample = all_features.sum(dim=1)
        return {
            "r_squared"              : 1 - total_mse / (total_var + 1e-8),
            "avg_reconstruction_mse" : total_mse / n,
            "num_alive_features"     : int(ever_active.sum().item()),
            "num_dead_features"      : num_dead,
            "pct_dead"               : num_dead / self.sae.d_hidden * 100,
            "avg_L0"                 : l0_per_sample.mean().item(),
            "L0_std"                 : l0_per_sample.std().item(),
            "total_features"         : self.sae.d_hidden,
        }


print("Classes defined: ActivationExtractor | SparseAutoencoder | SAETrainer | FeatureAnalyzer")

In [ ]:
# ── Build model ────────────────────────────────────────────────────────────

def load_state(model: nn.Module, ckpt_path: Path) -> dict:
    state = torch.load(ckpt_path, map_location="cpu")
    if "model_state_dict" in state:
        model.load_state_dict(state["model_state_dict"])
        return {k: v for k, v in state.items() if k != "model_state_dict"}
    if "state_dict" in state:
        cleaned = {k.replace("module.", "", 1): v for k, v in state["state_dict"].items()}
        model.load_state_dict(cleaned, strict=False)
        return {}
    cleaned = {k.replace("module.", "", 1): v for k, v in state.items()}
    model.load_state_dict(cleaned, strict=False)
    return {}


if MCFG["source"] == "torch_hub":
    from src.models.dinov2 import DINOv2Classifier
    model = DINOv2Classifier(num_classes=NUM_CLASSES, usage_mode="frozen_linear_probe")
else:
    model = timm.create_model(MCFG["timm_name"], pretrained=False, num_classes=NUM_CLASSES)

meta = load_state(model, CKPT_LOCAL)
model = model.to(DEVICE).eval()

print(f"Model            : {MODEL_NAME}")
print(f"Parameters       : {sum(p.numel() for p in model.parameters()):,}")
print(f"Checkpoint epoch : {meta.get('epoch', '?')}")
print(f"Checkpoint acc   : {meta.get('val_acc', '?')}")

In [ ]:
# ── Datasets ───────────────────────────────────────────────────────────────
# Training split → activation collection
# Test split     → feature analysis / steerability

crc_splits = CRCSplits(
    trainval_root=TRAINVAL_ROOT,
    test_root=TEST_ROOT,
    classes=tuple(CLASS_NAMES),
    val_ratio=0.25,
    random_state=SEED,
)

train_dataset = CRCHistologyDataset(
    split="train",
    splits=crc_splits,
    image_size=IMAGE_SIZE,
    return_id=False,
)
train_loader = DataLoader(
    train_dataset,
    batch_size=COLLECT_BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

test_dataset = CRCHistologyDataset(
    split="test",
    splits=crc_splits,
    image_size=IMAGE_SIZE,
    return_id=True,
)

# Balanced test subset for analysis / steerability
N_PER_CLASS_ANALYSIS = 30
class_counts: dict[int, int] = defaultdict(int)
analysis_indices: list[int] = []
for idx, label in enumerate(test_dataset.labels):
    lbl = int(label)
    if class_counts[lbl] < N_PER_CLASS_ANALYSIS:
        analysis_indices.append(idx)
        class_counts[lbl] += 1

analysis_dataset = Subset(test_dataset, analysis_indices)
analysis_loader  = DataLoader(
    analysis_dataset,
    batch_size=STEER_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
)

print(f"Train split   : {len(train_dataset):,} images ({len(train_loader)} batches)")
print(f"Test subset   : {len(analysis_indices)} images (balanced, {N_PER_CLASS_ANALYSIS}/class)")
print(f"Collecting from: {min(MAX_BATCHES_COLLECT, len(train_loader))} batches "
      f"≈ {min(MAX_BATCHES_COLLECT, len(train_loader)) * COLLECT_BATCH_SIZE:,} images")

n_tok = (
    MCFG["n_patch_tokens"] if TOKEN_SELECTION == "patches"
    else 1 if TOKEN_SELECTION == "cls"
    else MCFG["n_patch_tokens"] + MCFG["num_extra_tokens"]
)
est_tokens = min(MAX_BATCHES_COLLECT, len(train_loader)) * COLLECT_BATCH_SIZE * n_tok
print(f"Estimated tokens: {est_tokens:,} ({est_tokens * D_IN * 4 / 1e9:.2f} GB)")

In [ ]:
# ── Collect activations via ActivationExtractor ────────────────────────────
layers_str  = "_".join(map(str, EXTRACT_LAYERS))
cache_path  = (
    CACHE_DIR / f"acts_L{layers_str}_{EXTRACT_SITE}_{TOKEN_SELECTION}.pt"
    if CACHE_ACTS else None
)

if cache_path and cache_path.exists():
    print(f"Loading cached activations: {cache_path}")
    activations = torch.load(cache_path)
else:
    extractor = ActivationExtractor(
        model=model,
        model_name=MODEL_NAME,
        layers=EXTRACT_LAYERS,
        site=EXTRACT_SITE,
    )

    class _LimitedLoader:
        def __init__(self, loader, max_batches):
            self._loader    = loader
            self._max       = max_batches
        def __iter__(self):
            for i, batch in enumerate(self._loader):
                if i >= self._max:
                    break
                yield batch
        def __len__(self):
            return min(self._max, len(self._loader))

    acts_dict   = extractor.extract(
        _LimitedLoader(train_loader, MAX_BATCHES_COLLECT),
        token_mode=TOKEN_SELECTION,
        device=str(DEVICE),
    )
    # Use primary (first) layer activations
    primary_key = list(acts_dict.keys())[0]
    activations = acts_dict[primary_key]

    if cache_path:
        torch.save(activations, cache_path)
        print(f"Cached → {cache_path}")

print(f"\nActivations shape : {activations.shape}  (N_tokens × d_in)")
print(f"Memory usage       : {activations.nbytes / 1e9:.2f} GB")
print(f"Act mean           : {activations.mean():.4f}")
print(f"Act std            : {activations.std():.4f}")

sample = activations[torch.randperm(len(activations))[:10_000]].numpy()
plt.figure(figsize=(8, 3))
plt.hist(sample.flatten(), bins=100, log=True)
plt.xlabel("Activation value")
plt.ylabel("Count (log)")
plt.title(f"{MODEL_NAME} — layer {EXTRACT_LAYERS[0]}_{EXTRACT_SITE} — activation distribution (10k sample)")
plt.tight_layout()
plt.savefig(str(SAVE_DIR / "activation_distribution.png"), dpi=150)
plt.show()

In [ ]:
# ── Build and train SAE ────────────────────────────────────────────────────
set_seed(SEED)

sae = SparseAutoencoder(d_input=D_IN, d_sae=D_SAE, k=K_SPARSE)
print(f"SAE parameters : {sum(p.numel() for p in sae.parameters()):,}")
print(f"  W_enc        : {tuple(sae.W_enc.shape)}")
print(f"  W_dec        : {tuple(sae.W_dec.shape)}")
print(f"  aux_coeff    : {AUX_COEFF:.5f}  (ressuscite les features mortes)")

sae_cfg = {
    "lr"          : LEARNING_RATE,
    "batch_size"  : TRAIN_BATCH_SIZE,
    "num_epochs"  : NUM_EPOCHS,
    "warmup_steps": WARMUP_STEPS,
    "aux_coeff"   : AUX_COEFF,
}

trainer      = SAETrainer(sae, sae_cfg, device=str(DEVICE))
history_list = trainer.train(activations)

# Convert list[dict] → dict[list] for easy plotting
history = {k: [d[k] for d in history_list] for k in history_list[0]}

# Save checkpoint
sae_ckpt_path = SAVE_DIR / "sae_checkpoint.pth"
torch.save({
    "model_state_dict": sae.state_dict(),
    "d_in"            : D_IN,
    "d_sae"           : D_SAE,
    "k"               : K_SPARSE,
    "aux_coeff"       : AUX_COEFF,
    "extract_layers"  : EXTRACT_LAYERS,
    "extract_site"    : EXTRACT_SITE,
    "history"         : history,
}, sae_ckpt_path)
print(f"\nSAE checkpoint saved: {sae_ckpt_path}")

In [ ]:
# ── Training curves ────────────────────────────────────────────────────────
epochs = range(1, len(history["l2"]) + 1)

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
fig.suptitle(f"{MODEL_NAME} — SAE training — layer {EXTRACT_LAYERS[0]}_{EXTRACT_SITE}", fontsize=12)

axes[0].plot(epochs, history["l2"])
axes[0].set(title="Reconstruction loss (L2)", xlabel="Epoch", ylabel="Loss")
axes[0].grid(True)

axes[1].plot(epochs, history["aux"], color="tab:orange")
axes[1].set(title=f"Auxiliary loss (×{AUX_COEFF:.4f})\n[ressuscite features mortes]",
            xlabel="Epoch", ylabel="Loss")
axes[1].grid(True)

axes[2].plot(epochs, history["r2"], color="tab:red")
axes[2].axhline(y=0.90, color="green", linestyle="--", label="target R²=0.90")
axes[2].legend()
axes[2].set(title="Variance explained (R²)", xlabel="Epoch", ylabel="R²")
axes[2].set_ylim(0, 1)
axes[2].grid(True)

dead_pct = [d / D_SAE * 100 for d in history["dead"]]
axes[3].plot(epochs, dead_pct, color="tab:red")
axes[3].axhline(y=5, color="green", linestyle="--", label="target <5%")
axes[3].legend()
axes[3].set(title="Dead features (%)", xlabel="Epoch", ylabel="% dead")
axes[3].set_ylim(0, 100)
axes[3].grid(True)

plt.tight_layout()
plt.savefig(str(SAVE_DIR / "training_curves.png"), dpi=150)
plt.show()

print(f"\nFinal metrics:")
print(f"  L2 loss            : {history['l2'][-1]:.6f}")
print(f"  Aux loss           : {history['aux'][-1]:.6f}")
print(f"  L0                 : {history['l0'][-1]:.1f} / {K_SPARSE}")
print(f"  Variance explained : {history['r2'][-1]:.3f}")
print(f"  Dead features      : {int(history['dead'][-1])} / {D_SAE} ({history['dead'][-1]/D_SAE*100:.1f}%)")

In [ ]:
# ── SAE quality evaluation ─────────────────────────────────────────────────
sae.eval()

n_eval    = min(8192, len(activations))
eval_idx  = torch.randperm(len(activations))[:n_eval]
eval_acts = activations[eval_idx].to(DEVICE)

with torch.no_grad():
    x_hat, codes, l2_loss, _ = sae(eval_acts)

L0_per_token = (codes > 0).float().sum(-1).cpu().numpy()
recon_err    = l2_loss.item()
total_var    = F.mse_loss(eval_acts, eval_acts.mean(0, keepdim=True)).item()
var_exp      = max(0.0, 1.0 - recon_err / (total_var + 1e-8))
n_dead       = ((codes > 0).float().sum(0) == 0).sum().item()
dead_ratio   = n_dead / D_SAE

print("SAE quality on held-out 8192 tokens:")
print(f"  Reconstruction MSE   : {recon_err:.5f}")
print(f"  Variance explained   : {var_exp:.3f}  (target >0.90)")
print(f"  Mean L0              : {L0_per_token.mean():.1f}  (target {K_SPARSE})")
print(f"  Dead features        : {n_dead} / {D_SAE}  ({dead_ratio*100:.1f}%)  (target <5%)")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(L0_per_token, bins=50, edgecolor="black")
axes[0].axvline(x=K_SPARSE, color="red", linestyle="--", label=f"target k={K_SPARSE}")
axes[0].legend()
axes[0].set(title="L0 distribution per token", xlabel="L0", ylabel="Count")
axes[0].grid(True)

axes[1].bar(["Active", "Dead"], [D_SAE - n_dead, n_dead], color=["tab:green", "tab:red"])
axes[1].set(title=f"Feature activity ({D_SAE} total)", ylabel="Count")
for i, v in enumerate([D_SAE - n_dead, n_dead]):
    axes[1].text(i, v + 0.5, str(v), ha="center", fontweight="bold")
axes[1].grid(True, axis="y")

plt.tight_layout()
plt.savefig(str(SAVE_DIR / "sae_quality.png"), dpi=150)
plt.show()

with open(SAVE_DIR / "sae_metrics.json", "w") as f:
    json.dump({
        "model"              : MODEL_NAME,
        "extract_layers"     : EXTRACT_LAYERS,
        "extract_site"       : EXTRACT_SITE,
        "d_sae"              : D_SAE,
        "k"                  : K_SPARSE,
        "reconstruction_mse" : recon_err,
        "variance_explained" : var_exp,
        "mean_L0"            : float(L0_per_token.mean()),
        "n_dead_features"    : int(n_dead),
        "dead_feature_ratio" : dead_ratio,
    }, f, indent=2)

In [ ]:
# ── Collect test-subset activations for feature analysis ───────────────────
test_imgs_all: list[torch.Tensor] = []
test_lbls_all: list[int]          = []

for batch in analysis_loader:
    if len(batch) == 3:
        imgs, lbls, _ = batch
    else:
        imgs, lbls = batch
    test_imgs_all.append(imgs)
    test_lbls_all.extend(lbls.tolist())

test_imgs_tensor = torch.cat(test_imgs_all, dim=0)  # (N_test, C, H, W)
test_labels      = torch.tensor(test_lbls_all)
N_TEST           = len(test_imgs_tensor)
print(f"Test images: {N_TEST}")

layers_str       = "_".join(map(str, EXTRACT_LAYERS))
test_acts_cache  = (
    CACHE_DIR / f"test_acts_L{layers_str}_{EXTRACT_SITE}_{TOKEN_SELECTION}.pt"
    if CACHE_ACTS else None
)

if test_acts_cache and test_acts_cache.exists():
    print(f"Loading cached test activations: {test_acts_cache}")
    cached       = torch.load(test_acts_cache)
    test_acts    = cached["acts"]
    test_img_idx = cached["img_idx"]
else:
    class _LabeledLoader:
        def __init__(self, imgs, batch_size):
            self._imgs = imgs
            self._bs   = batch_size
        def __iter__(self):
            for i in range(0, len(self._imgs), self._bs):
                batch = self._imgs[i : i + self._bs]
                yield batch, torch.zeros(len(batch), dtype=torch.long)
        def __len__(self):
            return (len(self._imgs) + self._bs - 1) // self._bs

    test_extractor = ActivationExtractor(
        model=model, model_name=MODEL_NAME, layers=EXTRACT_LAYERS, site=EXTRACT_SITE
    )
    test_acts_dict = test_extractor.extract(
        _LabeledLoader(test_imgs_tensor, STEER_BATCH_SIZE),
        token_mode=TOKEN_SELECTION,
        device=str(DEVICE),
    )
    primary_key = list(test_acts_dict.keys())[0]
    test_acts   = test_acts_dict[primary_key]

    n_tok_per_img = (
        MCFG["n_patch_tokens"] if TOKEN_SELECTION == "patches"
        else 1 if TOKEN_SELECTION == "cls"
        else MCFG["n_patch_tokens"] + MCFG["num_extra_tokens"]
    )
    test_img_idx = torch.arange(N_TEST).repeat_interleave(n_tok_per_img)

    if test_acts_cache:
        torch.save({"acts": test_acts, "img_idx": test_img_idx}, test_acts_cache)

print(f"Test activations : {test_acts.shape}")
print(f"Image idx range  : {test_img_idx.min()}–{test_img_idx.max()}")

In [ ]:
# ── Analyse SAE features — top-K activating images per feature ─────────────
analyzer     = FeatureAnalyzer(sae=sae, device=str(DEVICE))
feature_data = analyzer.compute_top_activating(
    activations=test_acts,
    image_indices=test_img_idx,
    top_k_images=TOP_K_IMAGES_PER_FEATURE,
    batch_size=2048,
)

act_freqs = np.array([feature_data[i]["activation_frequency"] for i in range(D_SAE)])
mean_acts = np.array([feature_data[i]["mean_activation"]      for i in range(D_SAE)])

print(f"\nFeature statistics:")
print(f"  Active features (freq>0)  : {(act_freqs>0).sum()} / {D_SAE}")
print(f"  Mean activation frequency : {act_freqs.mean():.4f}")
print(f"  Max activation frequency  : {act_freqs.max():.4f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(act_freqs[act_freqs > 0], bins=50, log=True)
axes[0].set(title="Feature activation frequency", xlabel="Activation frequency", ylabel="Count (log)")
axes[0].grid(True)

axes[1].scatter(act_freqs, mean_acts, s=2, alpha=0.4)
axes[1].set(title="Mean activation vs frequency",
            xlabel="Activation frequency", ylabel="Mean activation value")
axes[1].grid(True)

plt.suptitle(f"{MODEL_NAME} — SAE feature analysis", fontsize=11)
plt.tight_layout()
plt.savefig(str(SAVE_DIR / "feature_statistics.png"), dpi=150)
plt.show()

# Select features to visualize: highest activation frequency, exclude ultra-common (freq>0.9)
freq_mask         = (act_freqs > 0) & (act_freqs < 0.9)
candidate_features = np.where(freq_mask)[0]
sorted_by_freq    = candidate_features[np.argsort(act_freqs[candidate_features])[::-1]]
vis_features      = sorted_by_freq[:N_FEATURES_VISUALIZE].tolist()
print(f"\nFeatures selected for visualization: {vis_features[:10]}… (top by freq)")

In [ ]:
# ── Feature visualization — top-K activating image montage ─────────────────

def denormalize(img_chw: torch.Tensor) -> np.ndarray:
    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
    img  = img_chw.detach().cpu().numpy().transpose(1, 2, 0)
    return np.clip(img * std + mean, 0, 1)


def get_class_distribution(image_idx_list: list[int]) -> dict[str, int]:
    counts: dict[str, int] = defaultdict(int)
    for img_idx in image_idx_list:
        if img_idx < len(test_labels):
            lbl = int(test_labels[img_idx])
            counts[CLASS_NAMES[lbl]] += 1
    return dict(counts)


feat_viz_dir = SAVE_DIR / "feature_montages"
feat_viz_dir.mkdir(exist_ok=True)

N_COLS  = 8
N_SHOW  = min(TOP_K_IMAGES_PER_FEATURE, N_COLS)

for feat_idx in vis_features[:N_FEATURES_VISUALIZE]:
    fdata  = feature_data[feat_idx]
    img_idx_list = fdata["top_image_indices"].tolist()
    strengths    = fdata["activation_strengths"].tolist()
    freq         = fdata["activation_frequency"]
    class_dist   = get_class_distribution(img_idx_list)

    n_show = min(N_SHOW, len(img_idx_list))
    fig, axes = plt.subplots(1, n_show, figsize=(2.5 * n_show, 3.0))
    if n_show == 1:
        axes = [axes]

    dominant_class = max(class_dist, key=class_dist.get) if class_dist else "?"
    fig.suptitle(
        f"Feature {feat_idx} | freq={freq:.3f} | dominant: {dominant_class}",
        fontsize=9,
    )

    for col_i, (img_idx, strength) in enumerate(zip(img_idx_list[:n_show], strengths[:n_show])):
        ax = axes[col_i]
        if img_idx < len(test_imgs_tensor):
            ax.imshow(denormalize(test_imgs_tensor[img_idx]))
            lbl = int(test_labels[img_idx]) if img_idx < len(test_labels) else -1
            cn  = CLASS_NAMES[lbl] if 0 <= lbl < NUM_CLASSES else "?"
            ax.set_title(f"{cn[:6]}\n{strength:.2f}", fontsize=7)
        else:
            ax.set_visible(False)
        ax.axis("off")

    plt.tight_layout(rect=[0, 0, 1, 0.90])
    out_path = feat_viz_dir / f"feat_{feat_idx:05d}.png"
    plt.savefig(str(out_path), dpi=120, bbox_inches="tight")
    plt.close()

print(f"Saved {N_FEATURES_VISUALIZE} feature montages → {feat_viz_dir}")

# Show first 6 in notebook
fig_all, axes_all = plt.subplots(2, 3, figsize=(15, 10))
for ax, feat_idx in zip(axes_all.flatten(), vis_features[:6]):
    img_path = feat_viz_dir / f"feat_{feat_idx:05d}.png"
    if img_path.exists():
        ax.imshow(Image.open(img_path))
    ax.set_title(f"Feature {feat_idx}", fontsize=9)
    ax.axis("off")
plt.suptitle(f"{MODEL_NAME} — Top features by activation frequency", fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# ── Feature × Class heatmap ────────────────────────────────────────────────
# Mean activation of each SAE feature per tissue class.

token_labels = test_labels[test_img_idx]  # (N_test_tokens,)

# FeatureAnalyzer batches the encoding internally
class_means_full = analyzer.feature_class_association(
    activations=test_acts,
    labels=token_labels,
    num_classes=NUM_CLASSES,
)  # (NUM_CLASSES, d_sae)

# Extract selected vis_features columns
n_vis             = len(vis_features)
class_feat_matrix = class_means_full[:, vis_features].numpy()  # (NUM_CLASSES, n_vis)

# Normalise columns for visibility
col_max                = class_feat_matrix.max(0, keepdims=True) + 1e-8
class_feat_matrix_norm = class_feat_matrix / col_max

fig, ax = plt.subplots(figsize=(min(n_vis * 0.5, 20), 5))
im = ax.imshow(class_feat_matrix_norm, aspect="auto", cmap="Reds")
ax.set_yticks(range(NUM_CLASSES))
ax.set_yticklabels(CLASS_NAMES, fontsize=8)
ax.set_xticks(range(n_vis))
ax.set_xticklabels([str(f) for f in vis_features], fontsize=6, rotation=90)
ax.set_xlabel("SAE feature index")
ax.set_ylabel("Class")
ax.set_title(f"{MODEL_NAME} — Feature × Class mean activation (normalized)")
plt.colorbar(im, ax=ax, label="Normalized mean activation")
plt.tight_layout()
plt.savefig(str(SAVE_DIR / "feature_class_heatmap.png"), dpi=150, bbox_inches="tight")
plt.show()

print("\nMost class-specific features (highest mean activation per class):")
for ci, cname in enumerate(CLASS_NAMES):
    best_local  = int(np.argmax(class_feat_matrix[ci]))
    best_global = vis_features[best_local]
    print(f"  {cname:20s}: feature {best_global}  (mean act = {class_feat_matrix[ci, best_local]:.4f})")

In [ ]:
# ── Steerability: ablate SAE features, measure class-probability shift ──────


def _get_module(model: nn.Module, path: str) -> nn.Module:
    """Navigate model hierarchy via dot-separated path."""
    m = model
    for part in path.split("."):
        m = getattr(m, part)
    return m


def steered_forward(
    model: nn.Module,
    images: torch.Tensor,
    sae: SparseAutoencoder,
    target_layer_path: str,
    feature_idx: int,
    mode: str = "ablate",
    strength: float = 0.0,
) -> torch.Tensor:
    """Forward pass with one SAE feature ablated or amplified via a hook."""
    module = _get_module(model, target_layer_path)

    def _hook(_mod, _inp, out: torch.Tensor) -> torch.Tensor:
        B, N_tok, D = out.shape
        flat     = out.reshape(-1, D)
        codes    = sae.encode(flat.to(DEVICE))
        residual = flat.to(DEVICE) - sae.decode(codes)
        if mode == "ablate":
            codes[:, feature_idx] = 0.0
        else:
            codes[:, feature_idx] *= strength
        patched = sae.decode(codes) + residual
        return patched.reshape(B, N_tok, D)

    handle = module.register_forward_hook(_hook)
    try:
        with torch.no_grad():
            out = model(images)
        if hasattr(out, "logits"):
            out = out.logits
        if isinstance(out, tuple):
            out = out[0]
    finally:
        handle.remove()
    return out


# Select top-5 features by class-specificity (max activation across classes minus mean)
specificity   = class_feat_matrix.max(0) - class_feat_matrix.mean(0)
specific_order = np.argsort(specificity)[::-1]
ablation_features = [vis_features[i] for i in specific_order[:5]]
print(f"Features selected for ablation study: {ablation_features}")

steer_imgs = test_imgs_tensor[:N_STEER_IMAGES].to(DEVICE)
steer_lbls = test_labels[:N_STEER_IMAGES].tolist()

# Base predictions
with torch.no_grad():
    base_out = model(steer_imgs)
    if hasattr(base_out, "logits"):
        base_out = base_out.logits
    if isinstance(base_out, tuple):
        base_out = base_out[0]
base_probs = torch.softmax(base_out, dim=1).cpu()

steer_results = []

for feat_idx in ablation_features:
    ablated_probs_list = []
    for start in range(0, N_STEER_IMAGES, STEER_BATCH_SIZE):
        batch_imgs = steer_imgs[start : start + STEER_BATCH_SIZE]
        logits = steered_forward(
            model=model, images=batch_imgs, sae=sae,
            target_layer_path=TARGET_LAYER_PATH,
            feature_idx=feat_idx, mode="ablate",
        )
        ablated_probs_list.append(torch.softmax(logits, dim=1).cpu())
    ablated_probs = torch.cat(ablated_probs_list, dim=0)

    pred_class        = base_probs.argmax(dim=1)
    base_pred_prob    = base_probs.gather(1, pred_class.unsqueeze(1)).squeeze(1)
    ablated_pred_prob = ablated_probs.gather(1, pred_class.unsqueeze(1)).squeeze(1)
    delta             = (ablated_pred_prob - base_pred_prob).numpy()

    steer_results.append({
        "feature_idx"       : feat_idx,
        "mean_delta_prob"   : float(delta.mean()),
        "std_delta_prob"    : float(delta.std()),
        "mean_base_prob"    : float(base_pred_prob.mean()),
        "mean_ablated_prob" : float(ablated_pred_prob.mean()),
    })
    print(f"  Feature {feat_idx:5d}: Δprob = {delta.mean():+.4f} ± {delta.std():.4f}")

fig, ax = plt.subplots(figsize=(8, 4))
feat_labels = [str(r["feature_idx"]) for r in steer_results]
deltas      = [r["mean_delta_prob"]  for r in steer_results]
colors      = ["tab:red" if d < 0 else "tab:green" for d in deltas]
ax.bar(feat_labels, deltas, color=colors, edgecolor="black")
ax.axhline(0, color="black", linewidth=0.8)
ax.set(title=f"{MODEL_NAME} — Ablation study: Δ predicted-class probability",
       xlabel="SAE feature index", ylabel="Δ prob (ablated − base)")
ax.grid(True, axis="y")
plt.tight_layout()
plt.savefig(str(SAVE_DIR / "steerability_ablation.png"), dpi=150)
plt.show()

with open(SAVE_DIR / "steerability_results.json", "w") as f:
    json.dump(steer_results, f, indent=2)

In [ ]:
# ── Steerability visualization on a single image ───────────────────────────
# Show original vs ablated prediction probability change as a bar chart overlay.

sample_idx = 0  # first test image
sample_img = steer_imgs[sample_idx : sample_idx + 1]
sample_lbl = int(test_labels[sample_idx])

with torch.no_grad():
    base_logit  = model(sample_img)
    if hasattr(base_logit, "logits"):
        base_logit = base_logit.logits
    if isinstance(base_logit, tuple):
        base_logit = base_logit[0]
    base_prob_all = torch.softmax(base_logit, dim=1).cpu().squeeze().numpy()

n_feats_show = min(5, len(ablation_features))
fig, axes = plt.subplots(1, n_feats_show + 1, figsize=(4 * (n_feats_show + 1), 5))

# Original image
axes[0].imshow(denormalize(steer_imgs[sample_idx].cpu()))
axes[0].set_title(f"GT: {CLASS_NAMES[sample_lbl]}", fontsize=9)
axes[0].axis("off")

for col_i, feat_idx in enumerate(ablation_features[:n_feats_show], start=1):
    ablated_logit = steered_forward(
        model, sample_img, sae, TARGET_LAYER, feat_idx, mode="ablate"
    )
    ablated_prob_all = torch.softmax(ablated_logit, dim=1).cpu().squeeze().numpy()
    delta_all = ablated_prob_all - base_prob_all

    ax = axes[col_i]
    colors = ["tab:red" if d < 0 else "tab:green" for d in delta_all]
    ax.barh(range(NUM_CLASSES), delta_all, color=colors)
    ax.set_yticks(range(NUM_CLASSES))
    ax.set_yticklabels(CLASS_NAMES, fontsize=7)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_title(f"Δ prob (feat {feat_idx} ablated)", fontsize=8)
    ax.set_xlabel("Δ probability", fontsize=7)
    ax.grid(True, axis="x")

plt.suptitle(f"{MODEL_NAME} — Ablation on one image", fontsize=11)
plt.tight_layout()
plt.savefig(str(SAVE_DIR / "ablation_single_image.png"), dpi=150)
plt.show()

In [ ]:
# ── Save feature analysis summary ─────────────────────────────────────────
feature_summary = []
for feat_idx in vis_features:
    fdata = feature_data[feat_idx]
    img_idx_list = fdata["top_image_indices"].tolist()
    class_dist   = get_class_distribution(img_idx_list)
    dominant     = max(class_dist, key=class_dist.get) if class_dist else None
    purity       = class_dist.get(dominant, 0) / max(sum(class_dist.values()), 1) if dominant else 0.0
    feature_summary.append({
        "feature_idx"         : feat_idx,
        "activation_frequency": fdata["activation_frequency"],
        "mean_activation"     : fdata["mean_activation"],
        "dominant_class"      : dominant,
        "purity"              : purity,
        "class_distribution"  : class_dist,
    })

with open(SAVE_DIR / "feature_summary.json", "w") as f:
    json.dump(feature_summary, f, indent=2)

# Print class-purity distribution
purities = np.array([r["purity"] for r in feature_summary])
print(f"Feature class-purity statistics (over {len(vis_features)} visualized features):")
print(f"  Mean purity  : {purities.mean():.3f}")
print(f"  Median purity: {np.median(purities):.3f}")
print(f"  >0.7 purity  : {(purities > 0.7).sum()} features")
print(f"  >0.9 purity  : {(purities > 0.9).sum()} features")

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(purities, bins=20, edgecolor="black")
ax.axvline(x=0.7, color="orange", linestyle="--", label="0.7")
ax.axvline(x=0.9, color="red",    linestyle="--", label="0.9")
ax.legend()
ax.set(title=f"{MODEL_NAME} — Feature class-purity distribution",
       xlabel="Class purity", ylabel="Number of features")
ax.grid(True)
plt.tight_layout()
plt.savefig(str(SAVE_DIR / "feature_purity.png"), dpi=150)
plt.show()

print(f"\nSaved feature summary: {SAVE_DIR / 'feature_summary.json'}")

In [ ]:
SAVE_DIR

In [ ]:
import json
# ── Upload results to Google Drive ─────────────────────────────────────────
RESULTS_DRIVE_FOLDER = DRIVE_FOLDER_ID  # same folder as checkpoint; update if needed

files_to_upload = [
    SAVE_DIR / "sae_checkpoint.pth",
    SAVE_DIR / "sae_metrics.json",
    SAVE_DIR / "training_curves.png",
    SAVE_DIR / "sae_quality.png",
    SAVE_DIR / "activation_distribution.png",
    SAVE_DIR / "feature_statistics.png",
    SAVE_DIR / "feature_class_heatmap.png",
    SAVE_DIR / "feature_purity.png",
    SAVE_DIR / "steerability_ablation.png",
    SAVE_DIR / "ablation_single_image.png",
    SAVE_DIR / "feature_summary.json",
    SAVE_DIR / "steerability_results.json",
]

# Also upload first N feature montages
montage_paths = sorted((SAVE_DIR / "feature_montages").glob("*.png"))[:20]
files_to_upload.extend(montage_paths)

for fpath in files_to_upload:
    fpath = Path(fpath)
    if not fpath.exists():
        print(f"  Not found (skipped): {fpath.name}")
        continue
    drive_file = drive.CreateFile({
        "title"  : f"{MODEL_NAME}_sae_{fpath.name}",
        "parents": [{"id": RESULTS_DRIVE_FOLDER}],
    })
    drive_file.SetContentFile(str(fpath))
    drive_file.Upload()
    print(f"  Uploaded: {MODEL_NAME}_sae_{fpath.name}")

print("\nDone.")

In [ ]:
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Cleanup done.")